# BassSpecMatchPRO — NAM no Google Colab
Upload direto do job `input.wav + output.wav`. Sem Cloudflare, token ou servidor intermediário.

In [ ]:
from google.colab import files
import pathlib, zipfile, json, wave, subprocess, shutil, time

def safe_extract(job, root):
    with zipfile.ZipFile(job) as z:
        for member in z.infolist():
            target = (root / member.filename).resolve()
            assert target == root.resolve() or root.resolve() in target.parents, "ZIP contém caminho inválido"
        z.extractall(root)

def wav_info(path):
    with wave.open(str(path), "rb") as w:
        info = (w.getframerate(), w.getnchannels(), w.getnframes(), w.getsampwidth())
        frames = w.readframes(w.getnframes())
    return info, frames

def validate_job(root):
    manifest_path = root / "manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError("manifest.json ausente")
    manifest = json.loads(manifest_path.read_text())
    if manifest.get("format") != "bassspec-nam-colab-job-v7" or manifest.get("schema_version") != 7:
        raise ValueError("job/schema Colab v7 inválido")
    if manifest.get("training_mode") != "input-output" or manifest.get("transport") != "colab-direct-upload":
        raise ValueError("contrato de treino/upload inválido")
    if manifest.get("reference_timbre_only") is not True or manifest.get("contains_source_audio") is not False:
        raise ValueError("proveniência do job inválida")
    if (manifest.get("reference_conditioned_target") is not True
        or manifest.get("canonical_output_authority") is not True
        or manifest.get("conditioned_transfer") is not True
        or manifest.get("linear_exports_from_same_target") is not True
        or manifest.get("target_model") != "reference-conditioned-fidelity-v2"
        or manifest.get("stimulus_role") != "carrier-only"
        or manifest.get("fidelity_fir_method") != "h1-wiener-8192-v1"):
        raise ValueError("job sem target condicionado compartilhado de fidelidade")
    if manifest.get("contains_reference_audio") is not False or (root / "reference.wav").exists():
        raise ValueError("reference.wav bruto não deve estar no job")
    for name in ("input.wav", "output.wav", "model.nam", "train_nam_official.py"):
        if not (root / name).is_file():
            if name == "output.wav":
                raise FileNotFoundError("output.wav ausente; fallback --reference proibido")
            raise FileNotFoundError(name + " ausente")
    input_info, input_bytes = wav_info(root / "input.wav")
    output_info, output_bytes = wav_info(root / "output.wav")
    if input_info != output_info:
        raise ValueError("input/output incompatíveis")
    if input_info[0] != 48000 or input_info[1] != 1 or input_info[2] <= 0:
        raise ValueError("input/output devem ser 48 kHz mono e não vazios")
    if not any(input_bytes) or not any(output_bytes):
        raise ValueError("input/output silencioso")
    if input_bytes == output_bytes:
        raise ValueError("output.wav não difere de input.wav")
    return manifest, input_info

print("BassSpecMatchPRO NAM — fila de jobs independentes")
print("Selecione um ou mais jobs ZIP do BassSpecMatchPRO")
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
assert zip_names, "Envie pelo menos um job ZIP do BassSpecMatchPRO"

subprocess.run(["pip", "install", "-q", "neural-amp-modeler==0.13.0"], check=True)
import torch
assert torch.cuda.is_available(), "Ative GPU T4/CUDA em Ambiente de execução > Alterar tipo de ambiente"
print("GPU:", torch.cuda.get_device_name(0))
print(f"Fila recebida: {len(zip_names)} job(s). Cada referência será treinada isoladamente.")

results = []
for job_index, zip_name in enumerate(zip_names, 1):
    job = pathlib.Path(zip_name)
    root = pathlib.Path(f"/content/bassspec_nam_job_{job_index}")
    shutil.rmtree(root, ignore_errors=True); root.mkdir()
    safe_extract(job, root)
    manifest, input_info = validate_job(root)
    epochs = int(manifest.get("epochs", manifest.get("default_epochs", 100)))
    assert 1 <= epochs <= 1000, "epochs fora do intervalo suportado (1..1000)"
    reference_name = str(manifest.get("reference_name") or job.stem).strip()
    reference_name = pathlib.Path(reference_name).name.strip() or f"Referencia_{job_index}"
    print(f"[{job_index}/{len(zip_names)}] {reference_name}: 48 kHz mono, {input_info[2]/48000:.1f}s, {epochs} epochs")

    preflight = pathlib.Path(f"/content/nam_preflight_{job_index}")
    shutil.rmtree(preflight, ignore_errors=True)
    cmd_preflight = ["python", str(root / "train_nam_official.py"), "--input", str(root / "input.wav"), "--output", str(root / "output.wav"), "--outdir", str(preflight), "--template", str(root / "model.nam"), "--preflight-only"]
    print("Preflight:", " ".join(cmd_preflight))
    subprocess.run(cmd_preflight, check=True)

    out = pathlib.Path(f"/content/nam_result_{job_index}")
    shutil.rmtree(out, ignore_errors=True); out.mkdir()
    log_path = out / "trainer.log"
    cmd = ["python", str(root / "train_nam_official.py"), "--input", str(root / "input.wav"), "--output", str(root / "output.wav"), "--outdir", str(out), "--template", str(root / "model.nam"), "--epochs", str(epochs), "--ignore-checks"]
    print("TREINO REAL:", " ".join(cmd))
    with log_path.open("w") as log_file:
        proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT, text=True)
        progress_path = out / "training-progress.json"
        last = None
        while proc.poll() is None:
            if progress_path.exists():
                try:
                    d = json.loads(progress_path.read_text())
                    stamp = (d.get("phase"), d.get("epoch"), d.get("epochs"), d.get("elapsed_seconds"))
                    if stamp != last:
                        print(f"[{job_index}/{len(zip_names)}] {d.get('phase')} | epoch {d.get('epoch')}/{d.get('epochs')} | {d.get('elapsed_seconds')}s")
                        last = stamp
                except Exception:
                    pass
            time.sleep(2)
    if proc.returncode != 0:
        raise RuntimeError(f"Falha no job {job.name}:\n" + log_path.read_text(errors="replace")[-5000:])

    model = out / "trained.nam"
    data = json.loads(model.read_text())
    assert data.get("architecture") == "SlimmableContainer", "trained.nam inválido"
    result_name = pathlib.Path("/content") / (reference_name + ".nam")
    shutil.copy2(model, result_name)
    results.append(result_name)
    print(f"[{job_index}/{len(zip_names)}] concluído: {result_name.name}")

print("Fila concluída. Baixando modelos individuais...")
for model in results:
    files.download(str(model))
